# HALO v2 — Evaluacion de Baselines (Naive + Media Movil + Estacional + **ARIMA**)
> **Taller de Sistemas Inteligentes** | Sprint 0

Evalua 4 modelos base. El umbral que Prophet debera superar en Sprint 1.

In [ ]:
import os, warnings
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import json
warnings.filterwarnings('ignore')

# ARIMA via statsmodels (puro Python, sin compilacion C++)
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': '#aaa',
    'ytick.color': '#aaa', 'grid.color': '#333',
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
})

NB_DIR = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.join(NB_DIR, '..')
DATA_FILE = os.path.join(ROOT_DIR, 'data', 'raw', 'synthetic_lapaz_daily.csv')
OUT_DIR   = os.path.join(ROOT_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_DAYS   = 292
HOLDOUT_DAYS = 73
PALETA = ['#e63946', '#2a9d8f', '#e9c46a', '#457b9d']
DIAS_SEMANA = ['Lun','Mar','Mie','Jue','Vie','Sab','Dom']
print('Librerias listas. ARIMA disponible via statsmodels.')

## 1. Carga de Datos y Test de Estacionariedad (ADF)

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f'Registros: {len(df):,} | Macrodistritos: {df["macrodistrito"].nunique()}')
print()

# Test de Dickey-Fuller aumentado para verificar estacionariedad
print('=== TEST DE ESTACIONARIEDAD (Dickey-Fuller) por Macrodistrito ===')
print(f'  p-value < 0.05 = Serie ESTACIONARIA (buena para ARIMA directo)')
print(f'  p-value > 0.05 = Necesita diferenciacion (d=1)')
print()
for zona in sorted(df['macrodistrito'].unique()):
    serie = df[df['macrodistrito'] == zona].sort_values('ds')['y'].values
    result = adfuller(serie, autolag='AIC')
    estatus = 'ESTACIONARIA' if result[1] < 0.05 else 'NO estacionaria'
    print(f'  {zona:<15}: p={result[1]:.4f}  -> {estatus}')

## 2. Funciones de Modelos y Metricas

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    if mask.sum() == 0: return float('nan')
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def compute_metrics(y_true, y_pred, model_name, zona):
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mape_v = mape(y_true, y_pred)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot != 0 else 0.0
    return {'model': model_name, 'zona': zona,
            'MAE': round(mae, 3), 'RMSE': round(rmse, 3),
            'MAPE_%': round(mape_v, 2), 'R2': round(r2, 3)}

def bl_naive(train, n): return np.full(n, train.iloc[-1])

def bl_moving_avg(train, n, w=7):
    return np.full(n, train.iloc[-w:].mean() if len(train) >= w else train.mean())

def bl_seasonal(train, holdout_dates):
    df_t = pd.DataFrame({'ds': train.index, 'y': train.values})
    df_t['wd'] = [datetime.strptime(x, '%Y-%m-%d').weekday() for x in df_t['ds']]
    means = df_t.groupby('wd')['y'].mean()
    preds = [means.get(datetime.strptime(d, '%Y-%m-%d').weekday(), df_t['y'].mean())
             for d in holdout_dates]
    return np.array(preds)

def bl_arima(train, n, order=(1,1,1)):
    """ARIMA via statsmodels. Puro Python, sin compilacion C++."""
    try:
        model = ARIMA(train.values, order=order)
        fit   = model.fit()
        fc    = fit.forecast(steps=n)
        return np.maximum(fc, 0)
    except Exception as e:
        print(f'    [ARIMA error] {e}')
        return np.full(n, train.mean())

print('Funciones de modelos definidas: Naive, Media Movil, Estacional, ARIMA(1,1,1)')

## 3. Evaluacion por Macrodistrito

In [ ]:
all_results = []
plot_data   = {}
zonas = sorted(df['macrodistrito'].unique())

for zona in zonas:
    print(f'  Evaluando: {zona}...')
    serie = df[df['macrodistrito'] == zona].sort_values('ds').set_index('ds')['y']
    if len(serie) < (TRAIN_DAYS + HOLDOUT_DAYS):
        print(f'    [SKIP] datos insuficientes')
        continue

    serie   = serie.iloc[-(TRAIN_DAYS + HOLDOUT_DAYS):]
    train   = serie.iloc[:TRAIN_DAYS]
    holdout = serie.iloc[TRAIN_DAYS:]
    y_true  = holdout.values
    n = len(holdout)

    p_naive    = bl_naive(train, n)
    p_ma7      = bl_moving_avg(train, n, 7)
    p_seasonal = bl_seasonal(train, holdout.index.tolist())
    p_arima    = bl_arima(train, n, order=(1, 1, 1))

    all_results.extend([
        compute_metrics(y_true, p_naive,    'BL-1 Naive',           zona),
        compute_metrics(y_true, p_ma7,      'BL-2 Media Movil 7d',  zona),
        compute_metrics(y_true, p_seasonal, 'BL-3 Estacional',      zona),
        compute_metrics(y_true, p_arima,    'BL-4 ARIMA(1,1,1)',    zona),
    ])

    plot_data[zona] = {
        'holdout': holdout,
        'preds': {
            'BL-1 Naive': p_naive,
            'BL-3 Estacional': p_seasonal,
            'BL-4 ARIMA': p_arima,
        }
    }

results_df = pd.DataFrame(all_results)
print('\nEvaluacion completada.')
results_df.head(8)

## 4. Tabla de Resultados — Comparativa Global

In [ ]:
summary = results_df.groupby('model')[['MAE','RMSE','MAPE_%','R2']].mean().round(3)
summary = summary.sort_values('RMSE')
summary.index.name = 'Modelo'

print('=== RESUMEN GLOBAL (promedio todas las zonas) ===')
print(summary.to_string())
print()
best = summary.iloc[0]
print(f'Mejor Baseline: {summary.index[0]}')
print(f'  MAE  = {best["MAE"]:.3f} incidentes/dia (error promedio)')
print(f'  RMSE = {best["RMSE"]:.3f}')
print(f'  MAPE = {best["MAPE_%"]:.1f}%')
print(f'  R2   = {best["R2"]:.3f}')
print()
print(f'Umbral Prophet (Sprint 1) debe superar:')
print(f'  RMSE < {best["RMSE"]*0.8:.3f} (20% de mejora minima)')
summary

## 5. Visualizacion — Comparacion de Modelos (Graficas de Barras)

In [ ]:
modelos = summary.index.tolist()
metricas = ['MAE', 'RMSE', 'MAPE_%', 'R2']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, metrica in zip(axes, metricas):
    vals = summary[metrica].values
    colors_bar = [PALETA[i % len(PALETA)] for i in range(len(modelos))]
    bars = ax.bar(range(len(modelos)), vals, color=colors_bar, width=0.6)
    ax.set_xticks(range(len(modelos)))
    ax.set_xticklabels([m.replace(' ', '\n') for m in modelos], fontsize=8)
    ax.set_title(f'{metrica}', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(vals)*0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='white')
    # Resaltar el mejor (menor para MAE/RMSE/MAPE, mayor para R2)
    best_idx = np.argmin(vals) if metrica != 'R2' else np.argmax(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2)

plt.suptitle('Comparacion de Modelos Baseline — HALO v2\n(Borde dorado = mejor modelo en cada metrica)', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'baseline_comparacion.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Prediccion vs Realidad — ARIMA y Estacional para Cada Zona

In [ ]:
n_zonas = len(plot_data)
fig, axes = plt.subplots(n_zonas, 1, figsize=(16, 4 * n_zonas), sharex=False)
if n_zonas == 1: axes = [axes]

for ax, zona in zip(axes, plot_data.keys()):
    holdout = plot_data[zona]['holdout']
    xs = [datetime.strptime(x, '%Y-%m-%d') for x in holdout.index]

    ax.plot(xs, holdout.values, label='Real', color='white', linewidth=2, zorder=5)
    ax.plot(xs, plot_data[zona]['preds']['BL-3 Estacional'],
            label='Estacional', color='#e9c46a', linestyle='--', linewidth=1.5)
    ax.plot(xs, plot_data[zona]['preds']['BL-4 ARIMA'],
            label='ARIMA(1,1,1)', color='#2a9d8f', linestyle=':', linewidth=2)

    ax.fill_between(xs, holdout.values, plot_data[zona]['preds']['BL-4 ARIMA'],
                    alpha=0.1, color='#2a9d8f', label='Error ARIMA')

    ax.set_title(f'Macrodistrito: {zona}   |   Periodo Holdout (73 dias)')
    ax.set_ylabel('Incidentes/dia')
    ax.legend(loc='upper right', framealpha=0.3, fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Prediccion vs Realidad — Baseline Estacional y ARIMA(1,1,1)\nHALO v2, Sprint 0', y=1.01, fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'baseline_pred_vs_real.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Guardado de Resultados (JSON y CSV)

In [ ]:
out_json = os.path.join(OUT_DIR, 'baseline_comparison.json')
out_csv  = os.path.join(OUT_DIR, 'baseline_comparison.csv')

results_df.to_csv(out_csv, index=False, encoding='utf-8')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump({
        'fecha': datetime.now().isoformat(),
        'resumen': summary.reset_index().to_dict(orient='records'),
        'detalle': all_results,
        'umbral_prophet': {
            'RMSE_objetivo': round(summary.iloc[0]["RMSE"] * 0.8, 3),
            'descripcion': '20% de mejora sobre el mejor baseline'
        }
    }, f, ensure_ascii=False, indent=2)

print(f'[OK] CSV  guardado: {out_csv}')
print(f'[OK] JSON guardado: {out_json}')
print()
print('=== FIN SPRINT 0 ===')
print('Siguiente paso: Sprint 1 -> Entrenar Prophet y comparar contra estos baselines.')